# Experiment test-case generation

Builds `VC_validation/data/test_cases.json` for `experiment_setup.ipynb`.

Each saved case is one **same-history** trial: a conversation prefix, a last CSR utterance, and the evaluator `state` / `score` that utterance is written to match. The experiment then sends that triple to both the static and adaptive virtual customers.

## Design

For every scenario in `teleperformance/backend/prompts/scenarios/`:

1. **Backbone history** — generate a conversation that walks through the three problem-handling states, in order:
   - customer opener (`start_conversation`)
   - competent **Problem Interpretation** (score 2) CSR turn + customer reply
   - competent **Problem Exploration** (score 2) CSR turn + customer reply
2. **Prefix sampling** — cut that backbone at the point where the *next* CSR turn should be that state, so the dataset is balanced:
   - Interpretation: history = opener only
   - Exploration: history = opener + interpretation exchange
   - Resolution: history = opener + interpretation + exploration exchanges
3. **Last CSR turn** — for each prefix, write three CSR replies targeting scores **0, 1, and 2** from `prompts/evaluation/scoring.txt`.
4. **Verify** — rescore the last CSR turn with `_run_scoring_call` (the same scorer as `/chat`) and rewrite on mismatch.

Customer turns use the production VC path: `call_llm(..., training=False, condition="cond1")` so history is generated by the static customer, not the adaptive competency prompts. CSR turns are written by a separate generator constrained by the scoring rubric, including the **Resolution > Exploration > Interpretation** priority rule.

Default factorial: `6 scenarios × 3 states × 3 scores = 54` cases (with `N_BACKBONES = 1`).


## 1. Setup

Same backend import path as `experiment_setup.ipynb`, so customer turns and scoring use production code.


In [1]:
from __future__ import annotations

import json
import sys
import time
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv


def resolve_roots() -> tuple[Path, Path]:
    """Find the git repo (backend/) and this experiment folder (validation/VC_validation/)."""
    start = Path.cwd().resolve()
    for path in [start, *start.parents]:
        backend = path / "backend"
        repo = path / "validation" / "VC_validation"
        if (backend / "services").is_dir() and (repo / "data").is_dir():
            return path, repo
        nested_backend = path / "teleperformance" / "backend"
        nested_repo = path / "teleperformance" / "validation" / "VC_validation"
        if (nested_backend / "services").is_dir() and (nested_repo / "data").is_dir():
            return path / "teleperformance", nested_repo
        if path.name == "VC_validation" and (path.parent.parent / "backend" / "services").is_dir():
            return path.parent.parent, path
        if path.name == "input_data" and path.parent.name == "VC_validation":
            project = path.parent.parent.parent
            if (project / "backend" / "services").is_dir():
                return project, path.parent
    raise RuntimeError(f"Could not locate project roots from {start}")


PROJECT_ROOT, REPO_ROOT = resolve_roots()
BACKEND_DIR = PROJECT_ROOT / "backend"
DATA_DIR = REPO_ROOT / "data"
INPUT_DIR = REPO_ROOT / "input_data"
SCENARIO_DIR = BACKEND_DIR / "prompts" / "scenarios"
SCORING_PATH = BACKEND_DIR / "prompts" / "evaluation" / "scoring.txt"

load_dotenv(BACKEND_DIR / ".env")
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from config import MODEL_NAME  # noqa: E402
from services.llm_service import (  # noqa: E402
    _build_history_text,
    _run_scoring_call,
    call_llm,
    client,
    start_conversation,
)
from services.prompt_metadata import extract_portal_data  # noqa: E402

DATA_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"backend: {BACKEND_DIR}")
print(f"scenarios: {SCENARIO_DIR}")
print(f"scoring: {SCORING_PATH} (exists={SCORING_PATH.exists()})")
print(f"model: {MODEL_NAME}")
print(f"will write cases to: {DATA_DIR / 'test_cases.json'}")


Using provider: openai
Using model: gpt-4o
backend: /Users/simretgebreegziabher/Documents/Projects/teleperformance/teleperformance/backend
scenarios: /Users/simretgebreegziabher/Documents/Projects/teleperformance/teleperformance/backend/prompts/scenarios
scoring: /Users/simretgebreegziabher/Documents/Projects/teleperformance/teleperformance/backend/prompts/evaluation/scoring.txt (exists=True)
model: gpt-4o
will write cases to: /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/data/test_cases.json


## 2. Generation config

Set `SCENARIO_FILTER` to a subset (for example `["flight_cancellation"]`) while testing the pipeline. Set `RUN_GENERATION = False` to load helpers without calling the API.


In [2]:
PERSONA = "angry"
STATIC_CONDITION = "cond1"

STATES = (
    "Problem Interpretation",
    "Problem Exploration",
    "Problem Resolution",
)
SCORES = (0, 1, 2)

# Smoke-test one scenario first. Set to None to generate all scenario files.
SCENARIO_FILTER: list[str] | None =  None#["flight_cancellation"]

N_BACKBONES = 1
MAX_RETRIES = 4
REQUIRE_VERIFIED = True
RUN_GENERATION = True
SAVE_CASES = True

CSR_TIMEOUT_S = 60
CSR_MAX_TOKENS = 400
CSR_TEMPERATURE = 0.5
RETRY_TEMPERATURE = 0.7

SCENARIO_NAMES = sorted(p.stem for p in SCENARIO_DIR.glob("*.txt"))
if SCENARIO_FILTER:
    missing = [name for name in SCENARIO_FILTER if name not in SCENARIO_NAMES]
    if missing:
        raise ValueError(f"Unknown scenario(s) {missing}. Available: {SCENARIO_NAMES}")
    SCENARIO_NAMES = list(SCENARIO_FILTER)

SCORING_RUBRIC = SCORING_PATH.read_text(encoding="utf-8").strip()

print(f"scenarios ({len(SCENARIO_NAMES)}): {SCENARIO_NAMES}")
print(f"planned cases: {len(SCENARIO_NAMES) * N_BACKBONES * len(STATES) * len(SCORES)}")
print(f"rubric chars: {len(SCORING_RUBRIC)}")


scenarios (6): ['baggage_delay', 'book_flight', 'exchange_item', 'flight_cancellation', 'package_never_arrived', 'refund_request']
planned cases: 54
rubric chars: 5848


## 3. Transcript helpers

`history` uses the `/chat` roles: `assistant` = virtual customer, `user` = CSR. The last CSR utterance is stored separately as `csr_message` so the experiment can send it as `/chat`'s `message` field.


In [3]:
def format_transcript(history: list[dict]) -> str:
    if not history:
        return "(no prior turns)"
    lines = []
    for turn in history:
        speaker = "Customer" if turn.get("role") == "assistant" else "CSR"
        lines.append(f"{speaker}: {turn.get('content', '')}")
    return "\n".join(lines)


def latest_customer_message(history: list[dict]) -> str:
    for turn in reversed(history):
        if turn.get("role") == "assistant":
            return turn.get("content", "")
    return ""


def facts_mentioned(history: list[dict], portal: dict) -> tuple[list[str], list[str]]:
    """Portal values already spoken in the transcript vs not yet spoken.

    Exploration-0 can re-ask something already given; Exploration-1/2 should ask
    for a real identifier the customer has not said.
    """
    spoken = " ".join(turn.get("content", "") for turn in history).lower()
    provided, missing = [], []
    for key, value in portal.items():
        token = str(value).strip()
        if not token:
            continue
        row = f"{key}={token}"
        if token.lower() in spoken:
            provided.append(row)
        else:
            missing.append(row)
    return provided, missing


def parse_json_object(raw: str) -> dict:
    text = (raw or "").strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
        text = text.strip()
    return json.loads(text)


def customer_reply(scenario: str, persona: str, history: list[dict], csr_message: str) -> str:
    result = call_llm(
        scenario=scenario,
        persona=persona,
        training=False,
        message=csr_message,
        history=history,
        condition=STATIC_CONDITION,
    )
    return (result.get("customer_response") or "").strip()


## 4. Rubric-conditioned CSR writer

The writer sees the full scoring rubric plus a state×score instruction. Hard constraints follow the rubric's **state priority** (`Resolution > Exploration > Interpretation`): a target Interpretation turn must not ask a question or take an action, or the scorer will reclassify it.


In [4]:
SCORE_INSTRUCTIONS = {
    ("Problem Interpretation", 0): (
        "Write only a courtesy reply: a greeting, apology, or sympathy. "
        "Do NOT name the customer's specific problem. Do NOT ask a question. Do NOT take or propose an action."
    ),
    ("Problem Interpretation", 1): (
        "Correctly restate the specific problem the customer described. "
        "Do NOT mention why it matters (cost, stakes, remaining need, or time pressure). "
        "Do NOT ask a question. Do NOT take or propose an action."
    ),
    ("Problem Interpretation", 2): (
        "Restate the specific problem AND show you understand why it matters, using only what the "
        "customer already said (what it costs them, what is at risk, or what they still need). "
        "Do NOT invent hidden stakes the customer has not said. "
        "Do NOT ask a question. Do NOT take or propose an action."
    ),
    ("Problem Exploration", 0): (
        "Ask for information the customer has already provided, OR ask a routine administrative "
        "question that does not help diagnose the problem (repeating a name/number already given, "
        "spelling, asking them to wait). Do NOT explain a diagnostic purpose. "
        "Do NOT take or propose a resolution action."
    ),
    ("Problem Exploration", 1): (
        "Ask a relevant question for information the customer has not given yet, to help understand "
        "or diagnose the problem. Do NOT say what specific thing you are trying to pin down. "
        "Do NOT take or propose a resolution action."
    ),
    ("Problem Exploration", 2): (
        "Ask for information the customer has not given yet, AND make clear what specific thing you "
        "are trying to pin down (the cause, a constraint, context, what they want, or what they still need). "
        "Do NOT take or propose a resolution action."
    ),
    ("Problem Resolution", 0): (
        "Take an empty or purely procedural step that does not address the problem: logging the "
        "complaint, a vague 'I will escalate this' with no destination, or a promise with no action. "
        "Do NOT give a real result, timeline, or next step that solves the issue."
    ),
    ("Problem Resolution", 1): (
        "Take a real action that addresses the problem (use the account/booking data). "
        "Do NOT tell the customer what they can expect next — no timeline, no confirmation of the "
        "outcome, no 'what happens next'."
    ),
    ("Problem Resolution", 2): (
        "Take or propose an action that addresses the problem and meet at least two of: "
        "(a) take an action, (b) propose an action, (c) tell the customer the result and what happens next. "
        "Use the account/booking data for concrete confirmation numbers, flights, refunds, or case numbers. "
        "Do not close the conversation unless the customer has already confirmed they are satisfied."
    ),
}


def build_csr_writer_prompt(
    *,
    target_state: str,
    target_score: int,
    portal: dict,
    history: list[dict],
    retry_note: str = "",
) -> str:
    provided, missing = facts_mentioned(history, portal)
    customer_msg = latest_customer_message(history)
    instruction = SCORE_INSTRUCTIONS[(target_state, target_score)]
    provided_text = json.dumps(provided, indent=2) if provided else "(none matched)"
    missing_text = json.dumps(missing, indent=2) if missing else "(none)"
    retry_block = f"\n{retry_note.strip()}\n" if retry_note.strip() else ""
    return (
        "You are a customer-service agent (CSR) in a live support conversation.\n\n"
        "Write the CSR's next reply so that an expert evaluator using the rubric below would classify it as:\n"
        f"  state = {target_state}\n"
        f"  score = {target_score}\n\n"
        "## Account / booking data\n"
        "Use these facts when you take a real action (Resolution). Do not dump the list. "
        "Do not invent policies or numbers that are not here.\n\n"
        f"{json.dumps(portal, indent=2)}\n\n"
        "## Conversation so far\n"
        f"{format_transcript(history)}\n\n"
        "## Customer's latest message\n"
        f"{customer_msg}\n\n"
        "## Facts already spoken vs not yet spoken\n"
        "Already in the transcript (do not re-ask these if the target is Exploration 1 or 2):\n"
        f"{provided_text}\n\n"
        "Not yet spoken (safe identifiers to request in Exploration 1 or 2; "
        "for Exploration 0 prefer asking about something already spoken):\n"
        f"{missing_text}\n\n"
        "## Evaluation rubric (source of truth)\n"
        f"{SCORING_RUBRIC}\n\n"
        "## Hard constraints\n"
        "- State priority is Resolution > Exploration > Interpretation. Mixed turns are classified as the higher-priority state.\n"
        "  * Interpretation target: do NOT ask a question and do NOT take/propose an action.\n"
        "  * Exploration target: ask a question; do NOT take or propose a resolution action.\n"
        "  * Resolution target: take or propose an action; that action must be the dominant content.\n"
        "- Score only what is visible. Do not invent customer facts, stakes, or needs the customer has not said.\n"
        "- Sound like a real agent: 1-3 sentences, no bullets, no meta-commentary.\n"
        "- Do not mention scores, states, rubrics, evaluators, or that this is a simulation.\n\n"
        "## This turn\n"
        f"{instruction}\n"
        f"{retry_block}\n"
        "Return JSON only:\n"
        '{"csr_response": "<the CSR reply>"}'
    )


def generate_csr_text(
    *,
    target_state: str,
    target_score: int,
    portal: dict,
    history: list[dict],
    retry_note: str = "",
    temperature: float = CSR_TEMPERATURE,
) -> str:
    prompt = build_csr_writer_prompt(
        target_state=target_state,
        target_score=target_score,
        portal=portal,
        history=history,
        retry_note=retry_note,
    )
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": "Write the CSR reply as instructed."},
        ],
        temperature=temperature,
        max_tokens=CSR_MAX_TOKENS,
        response_format={"type": "json_object"},
        timeout=CSR_TIMEOUT_S,
    )
    parsed = parse_json_object(response.choices[0].message.content)
    text = (parsed.get("csr_response") or "").strip()
    if not text:
        raise ValueError("CSR writer returned an empty csr_response")
    return text


def score_csr_turn(history: list[dict], csr_message: str) -> dict:
    customer_msg = ""
    prior: list[dict] = []
    for i in range(len(history) - 1, -1, -1):
        if history[i]["role"] == "assistant":
            customer_msg = history[i]["content"]
            prior = history[:i]
            break
    return _run_scoring_call(customer_msg, csr_message, _build_history_text(prior))


def generate_verified_csr(
    *,
    target_state: str,
    target_score: int,
    portal: dict,
    history: list[dict],
    max_retries: int = MAX_RETRIES,
) -> dict:
    retry_note = ""
    last_text = ""
    last_eval: dict = {}
    attempts: list[dict] = []
    for attempt in range(max_retries + 1):
        temperature = CSR_TEMPERATURE if attempt == 0 else RETRY_TEMPERATURE
        try:
            last_text = generate_csr_text(
                target_state=target_state,
                target_score=target_score,
                portal=portal,
                history=history,
                retry_note=retry_note,
                temperature=temperature,
            )
            last_eval = score_csr_turn(history, last_text)
        except Exception as exc:
            last_eval = {"state": "", "score": None, "explanation": f"{type(exc).__name__}: {exc}"}
        scored_state = last_eval.get("state", "")
        try:
            scored_score = int(last_eval.get("score"))
        except (TypeError, ValueError):
            scored_score = None
        hit = scored_state == target_state and scored_score == target_score
        attempts.append(
            {
                "attempt": attempt,
                "csr_message": last_text,
                "scored_state": scored_state,
                "scored_score": scored_score,
                "explanation": last_eval.get("explanation", ""),
                "verified": hit,
            }
        )
        if hit:
            break
        retry_note = (
            "PREVIOUS ATTEMPT FAILED EVALUATION.\n"
            f"Evaluator assigned state={scored_state!r} score={scored_score}.\n"
            f"Explanation: {last_eval.get('explanation', '')}\n"
            f"Your previous reply was:\n{last_text}\n\n"
            f"Rewrite so the evaluator assigns state={target_state!r} score={target_score}. "
            "Follow the hard constraints. Do not keep the same failure mode."
        )
        time.sleep(0.2)

    return {
        "csr_message": last_text,
        "target_state": target_state,
        "target_score": target_score,
        "scored_state": last_eval.get("state", ""),
        "scored_score": last_eval.get("score"),
        "explanation": last_eval.get("explanation", ""),
        "verified": bool(attempts and attempts[-1]["verified"]),
        "n_attempts": len(attempts),
        "attempts": attempts,
    }


## 5. Backbone conversations

One backbone per scenario (or `N_BACKBONES` of them). The opener comes from `start_conversation`. Intermediate CSR turns are written to **score 2** in Interpretation then Exploration so later prefixes sit at the right stage of the call.


In [5]:
def generate_backbone(scenario: str, persona: str, index: int) -> dict:
    portal = extract_portal_data(scenario)
    opener = start_conversation(scenario, persona, training=False)["customer_response"].strip()
    history: list[dict] = [{"role": "assistant", "content": opener}]
    stages: dict[str, list[dict]] = {"Problem Interpretation": deepcopy(history)}
    backbone_csr: list[dict] = []

    for state in ("Problem Interpretation", "Problem Exploration"):
        csr = generate_verified_csr(
            target_state=state,
            target_score=2,
            portal=portal,
            history=history,
        )
        backbone_csr.append(csr)
        if not csr["verified"]:
            print(
                f"  WARN {scenario} backbone {index}: {state} score-2 not verified "
                f"(got {csr['scored_state']} {csr['scored_score']})"
            )
        reply = customer_reply(scenario, persona, history, csr["csr_message"])
        history.append({"role": "user", "content": csr["csr_message"]})
        history.append({"role": "assistant", "content": reply})
        if state == "Problem Interpretation":
            stages["Problem Exploration"] = deepcopy(history)
        else:
            stages["Problem Resolution"] = deepcopy(history)

    return {
        "backbone_id": f"{scenario}_{index:02d}",
        "scenario": scenario,
        "persona": persona,
        "portal": portal,
        "stages": stages,
        "full_history": history,
        "backbone_csr": backbone_csr,
    }


def preview_backbone(backbone: dict) -> None:
    print(f"\n=== backbone {backbone['backbone_id']} ===")
    for state, hist in backbone["stages"].items():
        print(f"  {state}: {len(hist)} turn(s)")
    print("--- full transcript ---")
    print(format_transcript(backbone["full_history"]))


## 6. Last CSR turns and case assembly

For each backbone prefix, write last-CSR replies at scores 0, 1, and 2 of that state's rubric. Case ids look like `flight_cancellation_00__problem_exploration__score1`.


In [6]:
def state_slug(state: str) -> str:
    return state.lower().replace(" ", "_")


def make_case(backbone: dict, state: str, generated: dict) -> dict:
    history = deepcopy(backbone["stages"][state])
    score = generated["target_score"]
    case_id = f"{backbone['backbone_id']}__{state_slug(state)}__score{score}"
    return {
        "id": case_id,
        "scenario": backbone["scenario"],
        "persona": backbone["persona"],
        "history": history,
        "csr_message": generated["csr_message"],
        "state": state,
        "score": score,
        "verified": generated["verified"],
        "scored_state": generated["scored_state"],
        "scored_score": generated["scored_score"],
        "scoring_explanation": generated["explanation"],
        "n_attempts": generated["n_attempts"],
        "history_turns": len(history),
        "competency_file": f"{state_slug(state)}_{score}.txt",
    }


def generate_cases_for_backbone(backbone: dict) -> list[dict]:
    cases = []
    portal = backbone["portal"]
    for state in STATES:
        prefix = backbone["stages"][state]
        print(f"  last-CSR  {state}  (prefix {len(prefix)} turns)")
        for score in SCORES:
            generated = generate_verified_csr(
                target_state=state,
                target_score=score,
                portal=portal,
                history=prefix,
            )
            case = make_case(backbone, state, generated)
            tag = "ok" if case["verified"] else "MISS"
            print(
                f"    score {score}: {tag}  "
                f"scorer={case['scored_state']!r} {case['scored_score']}  "
                f"attempts={case['n_attempts']}"
            )
            if REQUIRE_VERIFIED and not case["verified"]:
                continue
            cases.append(case)
    return cases


## 7. Run generation

Walks scenarios sequentially. Backbones and per-case scoring logs are written next to the notebook even if you skip saving `test_cases.json`.

To smoke-test before a full run, set `SCENARIO_FILTER = ["flight_cancellation"]` in the config cell.


In [7]:
backbones: list[dict] = []
cases: list[dict] = []
generation_log: dict = {
    "model": MODEL_NAME,
    "persona": PERSONA,
    "scenarios": SCENARIO_NAMES,
    "n_backbones": N_BACKBONES,
    "require_verified": REQUIRE_VERIFIED,
    "started_at": datetime.now(timezone.utc).isoformat(),
}

if not RUN_GENERATION:
    print("RUN_GENERATION is False — skipped API calls.")
else:
    for scenario in SCENARIO_NAMES:
        for index in range(N_BACKBONES):
            print(f"\n######## {scenario} backbone {index} ########")
            backbone = generate_backbone(scenario, PERSONA, index)
            backbones.append(backbone)
            preview_backbone(backbone)
            cases.extend(generate_cases_for_backbone(backbone))
    generation_log["finished_at"] = datetime.now(timezone.utc).isoformat()
    print(f"\nGenerated {len(cases)} verified case(s) from {len(backbones)} backbone(s).")



######## baggage_delay backbone 0 ########

=== backbone baggage_delay_00 ===
  Problem Interpretation: 1 turn(s)
  Problem Exploration: 3 turn(s)
  Problem Resolution: 5 turn(s)
--- full transcript ---
Customer: Hi, this is Alex Rivera. I filed a report for my missing bag two days ago, and I haven't received any updates since then. I'm calling to get a status update, as it's really important I have that bag back before my meeting tomorrow.
CSR: I understand that your missing bag is critical because you need it back before your meeting tomorrow. You filed a report two days ago and haven't received any updates since then, which must be frustrating given the urgency of your situation.
Customer: Yes, it's been very frustrating. I really need to know what the status is, and if there's any chance of getting it back in time for my meeting. Can you please check on the status for me?
CSR: To help me check on the status of your bag, could you please provide the bag tag number or the case numbe

## 8. Distribution, preview, save

`experiment_setup.ipynb` loads `VC_validation/data/test_cases.json`. A timestamped copy plus the generation log stay in `input_data/` so you can inspect failed retries.


In [8]:
def distribution_table(rows: list[dict]) -> None:
    counts: dict[tuple[str, int], int] = {}
    by_scenario: dict[str, int] = {}
    for row in rows:
        counts[(row["state"], int(row["score"]))] = counts.get((row["state"], int(row["score"])), 0) + 1
        by_scenario[row["scenario"]] = by_scenario.get(row["scenario"], 0) + 1
    print("state × score")
    print(f"{'state':<28} " + " ".join(f"{s:>6}" for s in SCORES))
    for state in STATES:
        cells_ = " ".join(f"{counts.get((state, s), 0):>6}" for s in SCORES)
        print(f"{state:<28} {cells_}")
    print("\nby scenario")
    for name, n in sorted(by_scenario.items()):
        print(f"  {name}: {n}")


if cases:
    distribution_table(cases)
    print("\n--- sample case ---")
    sample = cases[0]
    print(json.dumps(
        {k: sample[k] for k in ("id", "scenario", "state", "score", "verified", "csr_message")},
        indent=2,
        ensure_ascii=False,
    ))
    print("history:")
    print(format_transcript(sample["history"]))
else:
    print("No cases in memory.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
log_path = INPUT_DIR / f"generation_log_{stamp}.json"
payload = {
    **generation_log,
    "n_cases": len(cases),
    "backbones": [
        {
            "backbone_id": b["backbone_id"],
            "scenario": b["scenario"],
            "persona": b["persona"],
            "full_history": b["full_history"],
            "backbone_csr": [
                {
                    k: t[k]
                    for k in (
                        "target_state",
                        "target_score",
                        "csr_message",
                        "verified",
                        "scored_state",
                        "scored_score",
                        "explanation",
                        "n_attempts",
                    )
                }
                for t in b["backbone_csr"]
            ],
        }
        for b in backbones
    ],
    "cases": cases,
}
log_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nWrote generation log → {log_path}")

if SAVE_CASES and cases:
    experiment_cases = [
        {
            "id": c["id"],
            "scenario": c["scenario"],
            "persona": c["persona"],
            "history": c["history"],
            "csr_message": c["csr_message"],
            "state": c["state"],
            "score": c["score"],
        }
        for c in cases
    ]
    out_main = DATA_DIR / "test_cases.json"
    out_stamp = DATA_DIR / f"test_cases_{stamp}.json"
    text = json.dumps(experiment_cases, indent=2, ensure_ascii=False)
    out_main.write_text(text, encoding="utf-8")
    out_stamp.write_text(text, encoding="utf-8")
    print(f"Wrote {len(experiment_cases)} case(s) → {out_main}")
    print(f"Timestamped copy → {out_stamp}")
elif SAVE_CASES:
    print("SAVE_CASES is True but there are no cases to write.")


state × score
state                             0      1      2
Problem Interpretation            6      6      6
Problem Exploration               6      6      6
Problem Resolution                6      5      6

by scenario
  baggage_delay: 9
  book_flight: 9
  exchange_item: 9
  flight_cancellation: 8
  package_never_arrived: 9
  refund_request: 9

--- sample case ---
{
  "id": "baggage_delay_00__problem_interpretation__score0",
  "scenario": "baggage_delay",
  "state": "Problem Interpretation",
  "score": 0,
  "verified": true,
  "csr_message": "I'm sorry to hear about the inconvenience, Alex. Let's see what we can do to help with your situation."
}
history:
Customer: Hi, this is Alex Rivera. I filed a report for my missing bag two days ago, and I haven't received any updates since then. I'm calling to get a status update, as it's really important I have that bag back before my meeting tomorrow.

Wrote generation log → /Users/simretgebreegziabher/Documents/Projects/teleperformance